In [ ]:
import os
from pathlib import Path
import shutil

import pandas as pd
import numpy as np
import functools

import teehr

import pyspark as ps
from pyspark.sql import functions as F

from assembly_utils import custom_event_detection as ced 

teehr.__version__

### Setup

In [ ]:
test_eval_dir = Path(Path().cwd(), "FIRO_test_evaluation")

# spark configuration
from teehr.evaluation.spark_session_utils import create_spark_session

'''
spark = create_spark_session(
    start_spark_cluster=True,
    executor_instances=2,
    executor_memory="50g",
    executor_cores=7
)

# access local evaluation
ev = teehr.LocalReadWriteEvaluation(
    dir_path=test_eval_dir,
    spark=spark
)
'''

ev = teehr.LocalReadWriteEvaluation(
    dir_path=test_eval_dir
)

ev.enable_logging()

### Create PF rank table

In [ ]:
def create_PF_event_rankings(
    quantile: str | None,
    configuration_toggle: str,
    variable_toggle: str,
) -> ps.sql.DataFrame:
    '''Creates event ranking tables'''

    print(f"Beginning calculations for quantile = {quantile}\n")

    # define the custom event detection instances
    custom_uniqueness_fields = [
        'primary_location_id',
        'configuration_name',
        'variable_name',
        'unit_name'
    ]
    
    q_10th = ced.AbovePercentileEventDetection_custom(
        quantile=0.1,
        add_quantile_field=True,
        add_event_peak_rank_field=True,
        uniqueness_fields=custom_uniqueness_fields
    )
    
    q_25th = ced.AbovePercentileEventDetection_custom(
        quantile=0.25,
        add_quantile_field=True,
        add_event_peak_rank_field=True,
        uniqueness_fields=custom_uniqueness_fields
    )
    
    q_50th = ced.AbovePercentileEventDetection_custom(
        quantile=0.5,
        add_quantile_field=True,
        add_event_peak_rank_field=True,
        uniqueness_fields=custom_uniqueness_fields
    )
    
    q_75th = ced.AbovePercentileEventDetection_custom(
        quantile=0.75,
        add_quantile_field=True,
        add_event_peak_rank_field=True,
        uniqueness_fields=custom_uniqueness_fields
    )
    
    q_90th = ced.AbovePercentileEventDetection_custom(
        quantile=0.9,
        add_quantile_field=True,
        add_event_peak_rank_field=True,
        uniqueness_fields=custom_uniqueness_fields
    )

    # create quantile dict to only apply defined quantile
    quantile_dict = {
        '10th':q_10th,
        '25th':q_25th,
        '50th':q_50th,
        '75th':q_75th,
        '90th':q_90th,
    }
    
    # define group_by list
    group_by_list = [
        "primary_location_id",
        "configuration_name",
        "variable_name",
        "event_above_id",
        "event_above_peak_rank"
    ]

    # define metrics list
    metrics_list = [
        teehr.Signatures.Maximum(
            output_field_name='peak_value'
        )
    ]

    # define filters list
    formatted_configs = ",".join([f"'{x}'" for x in configuration_toggle])
    filter_list = [
         f"configuration_name IN ({formatted_configs})",
         f"variable_name = '{variable_toggle}'",   
    ]

    # create JTS view & perform aggregation
    sdf = ev.joined_timeseries_view().add_calculated_fields([
        quantile_dict[quantile]
    ]).filter(
        filter_list
    ).aggregate(
        group_by=group_by_list,
        metrics=metrics_list
    ).order_by([
        'primary_location_id',
        'event_above_peak_rank'
    ]).to_sdf()

    # add a column to distinguish which quantile rankings correspond to
    sdf = sdf.withColumn('threshold', F.lit(f'{quantile}'))

    print(f"Finished calculations for quantile = {quantile}\n")

    return sdf

In [ ]:
# assemble peak flow rankings by quantile
quantiles = ['10th', '25th', '50th', '75th', '90th']
configuration_toggle = ["hefs_streamflow_forecast", "benchmark_forecast_hourly_normals"]
variable_toggle = "streamflow_hourly_inst"

result = {}
for quantile in quantiles:
    
    result[quantile] = create_PF_event_rankings(
        quantile=quantile,
        configuration_toggle=configuration_toggle,
        variable_toggle=variable_toggle
    )

rankings_sdf = functools.reduce(ps.sql.DataFrame.unionByName, result.values())

In [ ]:
# write to warehouse
ev._write.to_warehouse(source_data=rankings_sdf, table_name='event_rankings', write_mode='create_or_replace')

### Kill Spark

In [ ]:
ev.spark.stop()